In [1]:
import os
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig


In [2]:
finetuned_model_path = "/home/user/Desktop/PROJECT/llama/checkpoint-299"

# 8-bit quantization config
bnb_config = BitsAndBytesConfig(
    load_in_8bit=True,
    llm_int8_threshold=6.0,
    llm_int8_has_fp16_weight=False
)

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(finetuned_model_path, use_fast=False)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# Load model (CPU only to avoid OOM)
device = "cpu"

print("Loading model on CPU...")
model = AutoModelForCausalLM.from_pretrained(
    finetuned_model_path,
    quantization_config=bnb_config,
    device_map={"": device}
)

model.eval()
print("Model loaded successfully!")


Loading model on CPU...


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Model loaded successfully!


In [3]:
input_path = "/home/user/Desktop/PROJECT/PESConv.json"   # change here

print("Loading dataset from:", input_path)

if input_path.endswith(".jsonl"):
    data = [json.loads(line) for line in open(input_path, "r")]
else:
    data = json.load(open(input_path, "r"))

print("Total conversations:", len(data))


Loading dataset from: /home/user/Desktop/PROJECT/PESConv.json
Total conversations: 1300


In [ ]:
def build_prompt(sample):
    persona = sample.get("persona", "")

    dialog_text = ""
    for turn in sample["dialog"]:
        speaker = turn["speaker"]
        content = turn["content"].strip()
        dialog_text += f"{speaker}: {content}\n"

    prompt = f"""
### Instruction:
You are the SUPPORTER. Use the persona and conversation below.

Generate the *next supporter response* by completing **ALL FIVE SECTIONS**:
1. Emotion
2. Emotion Stimulus
3. Individual Appraisal
4. Strategy Reason
5. Response

You MUST output ALL 5 sections in this exact order.
Do NOT stop early.
Do NOT explain the task.
The final "Response:" must be ONE empathetic supporter message.

### Persona:
{persona}

### Conversation:
{dialog_text}

### OUTPUT FORMAT (STRICT):
Emotion: <short description>
Emotion Stimulus: <short description>
Individual Appraisal: <2–5 sentences>
Strategy Reason: <why the supporter chooses the strategy>
Response: <the actual supporter reply>

### Response:
"""

    return prompt.strip()


In [8]:
sample = data[0]
prompt = build_prompt(sample)

inputs = tokenizer(prompt, return_tensors="pt").to(device)

with torch.no_grad():
    output = model.generate(
        **inputs,
        max_new_tokens=300,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        repetition_penalty=1.2,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(output[0], skip_special_tokens=True)
print(response)


### Instruction:
You are the SUPPORTER. Use the persona and conversation below.

Generate the *next supporter response* by completing **ALL FIVE SECTIONS**:
1. Emotion
2. Emotion Stimulus
3. Individual Appraisal
4. Strategy Reason
5. Response

⚠️ You MUST output ALL 5 sections in this exact order.
⚠️ Do NOT stop early.
⚠️ Do NOT explain the task.
⚠️ The final "Response:" must be ONE empathetic supporter message.

### Persona:
<persona> my job is stressful but pays well. <persona> i work with people in hard financial situations. <persona> sometimes I wonder if it really is for me. <persona> my health and feelings come first. <persona> every year I get a big bonus. <persona> it makes me wonder if this is the right job for me anymore. <input>

### Conversation:
seeker: Hello
supporter: Hello, what would you like to talk about?
seeker: I am having a lot of anxiety about quitting my current job. It is too stressful but pays well
supporter: What makes your job stressful for you?
seeker: I ha

In [9]:
results = []

for i, sample in enumerate(data):
    print(f"Processing {i+1}/{len(data)}")

    prompt = build_prompt(sample)
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id
        )

    response = tokenizer.decode(output[0], skip_special_tokens=True)

    results.append({
        "original": sample,
        "cot_output": response
    })


Processing 1/1300
Processing 2/1300
Processing 3/1300
Processing 4/1300
Processing 5/1300
Processing 6/1300
Processing 7/1300
Processing 8/1300
Processing 9/1300
Processing 10/1300
Processing 11/1300
Processing 12/1300
Processing 13/1300
Processing 14/1300
Processing 15/1300
Processing 16/1300
Processing 17/1300
Processing 18/1300
Processing 19/1300
Processing 20/1300
Processing 21/1300
Processing 22/1300
Processing 23/1300
Processing 24/1300
Processing 25/1300
Processing 26/1300
Processing 27/1300
Processing 28/1300
Processing 29/1300
Processing 30/1300
Processing 31/1300
Processing 32/1300
Processing 33/1300
Processing 34/1300
Processing 35/1300
Processing 36/1300
Processing 37/1300
Processing 38/1300
Processing 39/1300
Processing 40/1300
Processing 41/1300
Processing 42/1300
Processing 43/1300
Processing 44/1300
Processing 45/1300
Processing 46/1300
Processing 47/1300
Processing 48/1300
Processing 49/1300
Processing 50/1300
Processing 51/1300
Processing 52/1300
Processing 53/1300
Pr

In [10]:
import json

# ✅ Path to save results
output_file = "/home/user/Desktop/PROJECT/cot_results.json"

# Save results
with open(output_file, "w", encoding="utf-8") as f:
    json.dump(results, f, ensure_ascii=False, indent=4)

print(f"Results saved to {output_file}")


Results saved to /home/user/Desktop/PROJECT/cot_results.json


In [ ]:
import torch

# Ensure model and tokenizer are loaded (from your previous code)
# model, tokenizer, device

print("Chatbot ready! Type 'exit' to quit.")

while True:
    # Take input from user
    user_input = input("\nYou: ")
    if user_input.lower() == "exit":
        break

    # Build a CoT-style prompt for the model
    prompt = f"""### Instruction:
You are the SUPPORTER. Generate the next supporter response for the following conversation using CoT (Emotion, Emotion Stimulus, Individual Appraisal, Strategy Reason, Response).

### Conversation:
seeker: {user_input}

### OUTPUT FORMAT (STRICT):
Emotion: <short description>
Emotion Stimulus: <short description>
Individual Appraisal: <2–5 sentences>
Strategy Reason: <why the supporter chooses the strategy>
Response: <the actual supporter reply>
"""

    # Tokenize
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # Generate output
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=300,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            repetition_penalty=1.2,
            pad_token_id=tokenizer.eos_token_id
        )

    # Decode
    response = tokenizer.decode(output[0], skip_special_tokens=True)
    
    print("\nModel:\n", response)
